# IdeaWeaver SLM Builder

Configure a from-scratch, Gemma-4-Nano-style small language model — interleaved local/global attention, grouped-query attention, QK-norm, partial RoPE, and cross-layer KV-cache sharing — and watch it actually train on TinyStories, with a real, live loss curve.

**To run this, press "Runtime" → "Run all" on a Tesla T4 Google Colab instance!** A GPU isn't strictly required (the backend falls back to CPU), but training is far too slow on CPU to be useful — use a T4 or better: *Runtime → Change runtime type → T4 GPU*.

[GitHub](https://github.com/ideaweaver-ai/ideaweaver-slm-builder) · [IdeaWeaver AI Labs](https://www.ideaweaver.ai) · [Building Small Language Models from Scratch (course)](https://www.ideaweaver.ai/courses)

> This notebook clones the repo, installs the Node.js frontend and Python training backend, starts both, and displays the app below as an embedded iframe — the same `serve_kernel_port_as_iframe` pattern Unsloth Studio uses. **"Start Training" is real**: it builds the exact model you configure and trains it on TinyStories, streaming real loss back into the chart. The first run downloads and tokenizes TinyStories (~10–20 minutes); every run after that reuses the cached tokenizer and data.

### Setup: clone the repo

In [ ]:
!git clone --depth 1 --branch main https://github.com/ideaweaver-ai/ideaweaver-slm-builder.git
%cd ideaweaver-slm-builder

### Install Node.js 20 and project dependencies

Colab's base image doesn't ship a recent-enough Node.js for Next.js 16, so this installs Node 20 from NodeSource first.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs > /dev/null 2>&1
!node -v && npm -v
!npm install --no-audit --no-fund

### Install the Python training backend's dependencies

Colab already has `torch` with CUDA support preinstalled — this deliberately does **not** touch `torch` or `numpy`, only adds what's missing (FastAPI, Uvicorn, SentencePiece, Hugging Face `datasets`), so the preinstalled CUDA-enabled PyTorch build is never reinstalled or downgraded.

In [ ]:
!pip install --quiet fastapi uvicorn sentencepiece datasets tqdm

### Start the training backend

Starts `train_service.py` (FastAPI) in the background on port 8001 and waits for its `/health` check to pass. The frontend (started in the next cell) talks to this over `http://127.0.0.1:8001` via its own API routes — the browser never calls this port directly.

In [ ]:
import subprocess, sys, threading, time, urllib.request

BACKEND_PORT = 8001

# Invoked as `python -m uvicorn` (not the bare `uvicorn` command) so it can't
# fail to launch just because pip's console-script bin dir isn't on PATH.
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "train_service:app", "--host", "127.0.0.1", "--port", str(BACKEND_PORT)],
    cwd="backend",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def _pump_backend():
    for line in backend_proc.stdout:
        print(line, end="")

threading.Thread(target=_pump_backend, daemon=True).start()

def _backend_healthy():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{BACKEND_PORT}/health", timeout=1) as r:
            return r.status == 200
    except Exception:
        return False

for _ in range(60):
    if backend_proc.poll() is not None:
        time.sleep(0.5)  # let the log-pumping thread flush the traceback above
        raise RuntimeError(
            f"Training backend process exited early (code {backend_proc.returncode}) — "
            "see the traceback printed above this cell for the real error."
        )
    if _backend_healthy():
        print("\nTraining backend is up on port", BACKEND_PORT)
        break
    time.sleep(1)
else:
    raise RuntimeError("Training backend didn't respond on /health in time — check the log output above.")

### Build and start the frontend

Builds a **production** Next.js bundle and starts it (not `next dev`) — dev mode's hot-reload
websocket doesn't survive being proxied through Colab's iframe or the tunnel below, and silently
breaks React entirely (clicks do nothing, the page never becomes interactive) even though the
page itself loads fine. Production mode has no such websocket, so this avoids that class of bug
outright. Takes a little longer to start (a real build step) but is what actually works here.

In [ ]:
import subprocess, threading, time

PORT = 3000

print("Building production frontend (this takes a bit longer than dev mode)...")
build = subprocess.run(["npm", "run", "build"], capture_output=True, text=True)
print(build.stdout[-2000:])
if build.returncode != 0:
    print(build.stderr[-2000:])
    raise RuntimeError("Frontend build failed — see the output above for the real error.")

proc = subprocess.Popen(
    ["npm", "run", "start", "--", "-p", str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

ready = threading.Event()

def _pump():
    for line in proc.stdout:
        print(line, end="")
        if "Ready in" in line:
            ready.set()

threading.Thread(target=_pump, daemon=True).start()

if not ready.wait(timeout=60):
    raise RuntimeError("Server didn't start in time — check the log output above.")

time.sleep(1)  # small buffer so the first request doesn't race the server

from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=1200, width="100%")

### (Optional) Get a public URL to open this outside Colab

The embedded iframe above already works without this. Run this cell only if you want a real, shareable `https://` link — e.g. to open the app in its own browser tab, or send it to someone else. It uses [`cloudflared`'s quick tunnels](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/) (no account or signup needed).

⚠️ The link is public and unauthenticated — anyone with it can use the app (including starting training and downloading checkpoints) for as long as this cell keeps running. Don't leave it running unattended or share it widely.

In [ ]:
!test -f cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared)

import re
import subprocess
import threading

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

url_found = threading.Event()
public_url = {"value": None}

def _pump_tunnel():
    for line in tunnel_proc.stdout:
        m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if m and not url_found.is_set():
            public_url["value"] = m.group(0)
            url_found.set()

threading.Thread(target=_pump_tunnel, daemon=True).start()

if url_found.wait(timeout=30):
    print(f"\n🌍 Public URL: {public_url['value']}")
    print("Stays live as long as this cell keeps running — don't restart the runtime or interrupt this cell.")
else:
    print("cloudflared didn't report a public URL in time. It may still be starting — check its output above, or re-run this cell.")

---

Click **▶ Start Training** in the app above — the first run will spend several minutes downloading and tokenizing TinyStories before the loss curve starts moving (watch the status line under the chart). You can **Stop** anytime and download the checkpoint it's saved so far.

Questions or bugs? Open an issue on [GitHub](https://github.com/ideaweaver-ai/ideaweaver-slm-builder/issues). Want to understand every one of these variables — attention internals, RoPE, KV caching, and the training loop — not just tune them? Check out [Building Small Language Models from Scratch](https://www.ideaweaver.ai/courses).